In [ ]:
import json
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from sentence_transformers import SentenceTransformer


In [ ]:
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

train_data = load_jsonl("/content/synthetic_data_for_classification (3).jsonl")
print("Train samples:", len(train_data))


Train samples: 1900


In [ ]:
def clean_text_list(texts):
    return [t if isinstance(t, str) else "" for t in texts]


In [ ]:
#extract train text and binary labels
train_anchor = clean_text_list([d["anchor_text"] for d in train_data])
train_a = clean_text_list([d["text_a"] for d in train_data])
train_b = clean_text_list([d["text_b"] for d in train_data])

y_train = [1 if d["text_a_is_closer"] == 1 else 0 for d in train_data]


In [ ]:
sum(x is None for x in train_anchor), \
sum(x is None for x in train_a), \
sum(x is None for x in train_b)


(0, 0, 0)

In [ ]:
#tf-idf features
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_anchor = tfidf.fit_transform(train_anchor)
tfidf_a = tfidf.transform(train_a)
tfidf_b = tfidf.transform(train_b)


In [ ]:
#tf-idf binary features
X_tfidf_train = np.hstack([
    np.abs(tfidf_anchor - tfidf_a).toarray(),
    np.abs(tfidf_anchor - tfidf_b).toarray()
])

print("TF-IDF train shape:", X_tfidf_train.shape)


TF-IDF train shape: (1900, 16000)


In [ ]:
#sbert features

#load data
sbert = SentenceTransformer("all-MiniLM-L6-v2")

#encode train text
anchor_emb = sbert.encode(train_anchor, batch_size=32, show_progress_bar=True)
a_emb = sbert.encode(train_a, batch_size=32, show_progress_bar=True)
b_emb = sbert.encode(train_b, batch_size=32, show_progress_bar=True)


#sbert binary feature


X_sbert_train = np.hstack([
    np.abs(anchor_emb - a_emb),
    np.abs(anchor_emb - b_emb)
])

print("SBERT train shape:", X_sbert_train.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

SBERT train shape: (1900, 768)


In [ ]:
#concatening both

X_train = np.hstack([
    X_tfidf_train,   # lexical features
    X_sbert_train    # semantic features
])

# columns = TF-IDF features + SBERT features
print("Combined train shape:", X_train.shape)


Combined train shape: (1900, 16768)


X_tfidf_train:

 Word / n-gram based features capturing lexical overlap between texts
 (surface-level similarity using TF-IDF)

 X_sbert_train:

 Semantic distance-based features from SBERT embeddings
 (captures meaning-level similarity independent of exact words)


 np.hstack:

 Horizontally concatenate TF-IDF and SBERT features
 so each sample has both lexical and semantic information



Combined feature matrix:
 rows = samples
 columns = TF-IDF features + SBERT features


In [ ]:
#trsin classifier
clf=LogisticRegression(max_iter=1000)
clf.fit(X_train,y_train)
print("combined model trained")

combined model trained


In [ ]:
#testing on dev data

dev_data = load_jsonl("/content/dev_track_a.jsonl")

#extract dev texts
dev_anchor = [d["anchor_text"] for d in dev_data]
dev_a = [d["text_a"] for d in dev_data]
dev_b = [d["text_b"] for d in dev_data]

y_dev = [1 if d["text_a_is_closer"] == 1 else 0 for d in dev_data]

#tf idf dev feaatures
tfidf_anchor_dev = tfidf.transform(dev_anchor)
tfidf_a_dev = tfidf.transform(dev_a)
tfidf_b_dev = tfidf.transform(dev_b)

X_tfidf_dev = np.hstack([
    np.abs(tfidf_anchor_dev - tfidf_a_dev).toarray(),
    np.abs(tfidf_anchor_dev - tfidf_b_dev).toarray()
])

#sbert dev features
anchor_emb_dev = sbert.encode(dev_anchor, batch_size=32, show_progress_bar=True)
a_emb_dev = sbert.encode(dev_a, batch_size=32, show_progress_bar=True)
b_emb_dev = sbert.encode(dev_b, batch_size=32, show_progress_bar=True)

X_sbert_dev = np.hstack([
    np.abs(anchor_emb_dev - a_emb_dev),
    np.abs(anchor_emb_dev - b_emb_dev)
])

#combine dev features
X_dev = np.hstack([
    X_tfidf_dev,
    X_sbert_dev
])

print("Combined dev shape:", X_dev.shape)

#predicting final metrics
y_pred = clf.predict(X_dev)

print("Accuracy:", accuracy_score(y_dev, y_pred))
print(classification_report(y_dev, y_pred))




Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Combined dev shape: (200, 16768)
Accuracy: 0.575
              precision    recall  f1-score   support

           0       0.57      0.60      0.58        99
           1       0.58      0.55      0.57       101

    accuracy                           0.57       200
   macro avg       0.58      0.58      0.57       200
weighted avg       0.58      0.57      0.57       200

